# 🏥 Stroke Prediction: Real-World Inference 

In the primary tutorial notebook (`stroke_prediction_tutorial.ipynb`), we acted as Data Scientists:
1. We explored historical data (EDA).
2. We engineered new features (BMI/Age Categories, High Glucose Flags).
3. We built a robust scikit-learn `Pipeline` to handle missing values and One-Hot Encoding.
4. We used SMOTE to handle imbalanced data and GridSearch to tune an XGBoost model.
5. Most importantly, **we saved the entire pipeline to a file**: `models/stroke_pipeline_v1.pkl`.

**Now, we are acting as AI Engineers.** 
This notebook simulates what happens in production (e.g., behind a web app or API). We don't need to rebuild the model. We just load it, give it raw patient data, and get a prediction out!

## 1. Load the Pre-Trained Pipeline
We use `joblib` to bring our saved model back to life. Notice how fast this is compared to training it from scratch!

In [1]:
import joblib
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 🚨 Critical for Unpickling: 
# When saving a custom function (like our 'categorize_features') inside an sklearn Pipeline, 
# python needs to know what that function is before it can load it!
def categorize_features(df_in):
    df = df_in.copy()
    
    df['is_high_glucose'] = (df['avg_glucose_level'] > 150).astype(int)
    
    bins = [0, 18.5, 25, 30, 100]
    labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
    df['bmi_category'] = pd.cut(df['bmi'].fillna(df['bmi'].median()), bins=bins, labels=labels)
    
    age_bins = [0, 18, 65, 150]
    age_labels = ['Child', 'Adult', 'Senior']
    df['age_category'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)
    
    df['health_risk_score'] = (
        (df['age_category'] == 'Senior').astype(int) +
        df['hypertension'] +
        df['heart_disease'] +
        (df['bmi_category'].isin(['Overweight', 'Obese'])).astype(int) +
        (df['smoking_status'] == 'smokes').astype(int)
    )
    return df

# Load the entire pipeline (cleaning + feature engineering + XGBoost model)
try:
    production_model = joblib.load('models/stroke_pipeline_v1.pkl')
    print("✅ Model successfully loaded and ready for predictions!")
except FileNotFoundError:
    print("❌ Error: Could not find 'models/stroke_pipeline_v1.pkl'.")
    print("Did you run all the cells in 'stroke_prediction_tutorial.ipynb' first to generate it?")

✅ Model successfully loaded and ready for predictions!


## 2. Simulating "New" Patient Data
information for three brand new patients into a hospital database.

Notice that this data is **completely raw**. It has missing BMI values, it has 'Yes'/'No' instead of 1/0, and it doesn't have our custom features like `health_risk_score`.

Because we built a `Pipeline` earlier, we don't need to manually clean this data! The pipeline remembers exactly what to do.

In [ ]:
# Create a DataFrame simulating raw input from a web form
new_patients = pd.DataFrame([
    {
        'gender': 'Male',
        'age': 78.0,
        'hypertension': 1,
        'heart_disease': 1,
        'ever_married': 'Yes',
        'work_type': 'Private',
        'Residence_type': 'Urban',
        'avg_glucose_level': 210.5,
        'bmi': 36.6,
        'smoking_status': 'smokes'
    },
    {
        # Patient B: Healthy Profile (Young, active, normal vitals)
        'gender': 'Female',
        'age': 25.0,
        'hypertension': 0,
        'heart_disease': 0,
        'ever_married': 'No',
        'work_type': 'Private',
        'Residence_type': 'Rural',
        'avg_glucose_level': 85.0,
        'bmi': 22.4, # Healthy BMI
        'smoking_status': 'never smoked'
    },
    {
        # Patient C: Missing Data Profile (To prove our Imputer works!)
        'gender': 'Male',
        'age': 55.0,
        'hypertension': 0,
        'heart_disease': 0,
        'ever_married': 'Yes',
        'work_type': 'Self-employed',
        'Residence_type': 'Suburban', # Note: Our pipeline might not have seen 'Suburban' exactly, but it should handle it gracefully or ignore it.
        'avg_glucose_level': 130.0,
        'bmi': None, # Missing BMI! Our pipeline's SimpleImputer will fill this with the median it learned during training.
        'smoking_status': 'Unknown'
    }
])

display(new_patients)

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
0,Male,78.0,1,1,Yes,Private,Urban,210.5,36.6,smokes
1,Female,25.0,0,0,No,Private,Rural,85.0,22.4,never smoked
2,Male,55.0,0,0,Yes,Self-employed,Suburban,130.0,NaN,Unknown


## 3. One-Click Inference
Normally, we would have to write hundreds of lines of code to clean `new_patients` before giving it to an XGBoost model. 

But watch what happens when we pass raw data directly into our loaded pipeline:

In [3]:
# 1. Ask the pipeline to predict Stroke (1) or No Stroke (0)
predictions = production_model.predict(new_patients)

# 2. Ask the pipeline for the probability (confidence) behind those predictions
# [:, 1] grabs the probability of Class 1 (Stroke)
probabilities = production_model.predict_proba(new_patients)[:, 1]

# 3. Add the results to our mock database for easy reading
results_df = new_patients.copy()
results_df['Predicted_Stroke_Risk'] = ['High Risk 🚨' if p == 1 else 'Low Risk ✅' for p in predictions]
results_df['Probability_Score'] = [f"{prob * 100:.1f}%" for prob in probabilities]

# Show the doctor the final output!
display(results_df[['age', 'avg_glucose_level', 'Predicted_Stroke_Risk', 'Probability_Score']])

,age,avg_glucose_level,Predicted_Stroke_Risk,Probability_Score
0,78.0,210.5,Low Risk ✅,49.6%
1,25.0,85.0,Low Risk ✅,31.0%
2,55.0,130.0,Low Risk ✅,45.5%


### 🎉 Mission Accomplished
You went all the way from messy CSV data to a deployable Machine Learning system. 

The integration of `sklearn.pipeline.Pipeline` is what allowed us to clean `Patient C`'s missing BMI and `Patient A`'s categorical columns completely automatically during inference. This is how real AI systems are built into applications!